## Imports + config

In [ ]:
import pandas as pd
# Initialize to keep the IDE type-checker happy
elo = pd.DataFrame()
from pathlib import Path
import sys
import importlib
import os

# Navigate to project root and update sys.path
# The notebook is in ./notebooks/, so parent gets us to project root
os.chdir(r'c:\Github\fifa2026-prediction')
PROJECT_ROOT = Path.cwd()
print(f"Project root: {PROJECT_ROOT}")
print(f"Current working directory: {os.getcwd()}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import and reload to get latest changes
import src.data_loader
importlib.reload(src.data_loader)
from src.data_loader import load_raw_matches, load_raw_elo, save_processed

# Store root for later use
root = PROJECT_ROOT

Project root: c:\Github\fifa2026-prediction
Current working directory: c:\Github\fifa2026-prediction


## Load raw data

In [26]:
# Fetch real international football data from football-data.co.uk
import pandas as pd
from pathlib import Path
import urllib.request
import io

print("Fetching real international football match data...")

# Source: football-data.co.uk provides historical international match results
url = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"

try:
    # Try to fetch real data
    with urllib.request.urlopen(url, timeout=10) as response:
        matches = pd.read_csv(io.StringIO(response.read().decode('utf-8')))
    
    print(f"✓ Real data loaded from football-data.co.uk")
    print(f"  Shape: {matches.shape}")
    print(f"  Columns: {matches.columns.tolist()}")
    print(f"\nFirst few rows:")
    print(matches.head())
    
except Exception as e:
    print(f"⚠ Could not fetch real data online: {e}")
    print("Creating sample data instead...")
    
    # Fallback to sample data
    matches_data = {
        'Date': ['2023-01-01', '2023-01-02', '2023-01-03', '2023-01-04', '2023-01-05'],
        'Home': ['Brazil', 'Germany', 'France', 'Spain', 'England'],
        'Away': ['Argentina', 'Italy', 'Belgium', 'Portugal', 'Netherlands'],
        'HG': [2, 1, 3, 2, 1],
        'AG': [1, 1, 1, 0, 2]
    }
    matches = pd.DataFrame(matches_data)
    print(f"Sample Matches data created:")
    print(matches)

# Sample ELO ratings data
elo_data = {
    'team': ['Brazil', 'Germany', 'France', 'Spain', 'Portugal', 'Argentina', 'Belgium', 'Italy', 'England', 'Netherlands'],
    'elo_rating': [1840, 1780, 1760, 1715, 1710, 1705, 1695, 1680, 1670, 1660]
}

elo = pd.DataFrame(elo_data)

print("\n\nELO Ratings data:")
print(elo)
print(f"\nShape: {elo.shape}")
print(f"Columns: {elo.columns.tolist()}")

Fetching real international football match data...
✓ Real data loaded from football-data.co.uk
  Shape: (49368, 9)
  Columns: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']

First few rows:
         date home_team away_team  home_score  away_score tournament     city  \
0  1872-11-30  Scotland   England         0.0         0.0   Friendly  Glasgow   
1  1873-03-08   England  Scotland         4.0         2.0   Friendly   London   
2  1874-03-07  Scotland   England         2.0         1.0   Friendly  Glasgow   
3  1875-03-06   England  Scotland         2.0         2.0   Friendly   London   
4  1876-03-04  Scotland   England         3.0         0.0   Friendly  Glasgow   

    country  neutral  
0  Scotland    False  
1   England    False  
2  Scotland    False  
3   England    False  
4  Scotland    False  


ELO Ratings data:
          team  elo_rating
0       Brazil        1840
1      Germany        1780
2       France        17

## Clean + target creation

In [27]:
# Standardize column names and add result column
def standardize_and_add_result(df):
    """
    Standardize column names and add result column
    Result: 1 = home win, 0 = draw or away win
    """
    df = df.copy()
    
    # Standardize column names (handle various naming conventions)
    col_mapping = {
        'Date': 'date', 'date': 'date',
        'Home': 'home_team', 'home': 'home_team', 'home_team': 'home_team',
        'Away': 'away_team', 'away': 'away_team', 'away_team': 'away_team',
        'HG': 'home_goals', 'HomeGoals': 'home_goals', 'home_goals': 'home_goals',
        'AG': 'away_goals', 'AwayGoals': 'away_goals', 'away_goals': 'away_goals',
    }
    
    # Rename columns that match mapping
    rename_dict = {old: new for old, new in col_mapping.items() if old in df.columns}
    df.rename(columns=rename_dict, inplace=True)
    
    print(f"Standardized columns: {df.columns.tolist()}")
    
    # Add result column
    df["result"] = 0
    if 'home_goals' in df.columns and 'away_goals' in df.columns:
        df.loc[df["home_goals"] > df["away_goals"], "result"] = 1
        print(f"✓ Result column added (1=home win, 0=draw/loss)")
    else:
        print(f"⚠ Could not find goal columns to compute result")
    
    return df

matches = standardize_and_add_result(matches)
print(f"\n✓ Data preprocessing complete!")
print(f"  Matches shape: {matches.shape}")
print(f"  Columns: {matches.columns.tolist()}\n")
print(matches.head())

Standardized columns: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']
⚠ Could not find goal columns to compute result

✓ Data preprocessing complete!
  Matches shape: (49368, 10)
  Columns: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'result']

         date home_team away_team  home_score  away_score tournament     city  \
0  1872-11-30  Scotland   England         0.0         0.0   Friendly  Glasgow   
1  1873-03-08   England  Scotland         4.0         2.0   Friendly   London   
2  1874-03-07  Scotland   England         2.0         1.0   Friendly  Glasgow   
3  1875-03-06   England  Scotland         2.0         2.0   Friendly   London   
4  1876-03-04  Scotland   England         3.0         0.0   Friendly  Glasgow   

    country  neutral  result  
0  Scotland    False       0  
1   England    False       0  
2  Scotland    False       0  
3   England    False  

In [28]:
# Helper function to load real data when you add CSV files to data/raw/
def load_data_from_csv():
    """
    Load real data from CSV files in data/raw/
    Expects: international_matches.csv and elo_ratings.csv
    """
    raw_dir = root / "data" / "raw"
    
    matches_file = raw_dir / "international_matches.csv"
    elo_file = raw_dir / "elo_ratings.csv"
    
    matches, elo = None, None
    
    # Check if files exist
    if matches_file.exists():
        print(f"✓ Found matches file: {matches_file}")
        try:
            # Try multiple encodings
            for encoding in ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252']:
                try:
                    matches = pd.read_csv(matches_file, encoding=encoding)
                    print(f"  ✓ Loaded with {encoding} encoding")
                    print(f"  Shape: {matches.shape}")
                    break
                except:
                    continue
        except Exception as e:
            print(f"  ⚠ Could not load: {e}")
            print(f"  File may be corrupted or in wrong format")
    else:
        print(f"⚠ {matches_file.name} not found")
    
    if elo_file.exists():
        print(f"✓ Found ELO file: {elo_file}")
        try:
            for encoding in ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252']:
                try:
                    elo = pd.read_csv(elo_file, encoding=encoding)
                    print(f"  ✓ Loaded with {encoding} encoding")
                    print(f"  Shape: {elo.shape}")
                    break
                except:
                    continue
        except Exception as e:
            print(f"  ⚠ Could not load: {e}")
    else:
        print(f"⚠ {elo_file.name} not found")
    
    return matches, elo

# Try loading real data
print("Checking for real data files...\n")
real_matches, real_elo = load_data_from_csv()

# Use real data if found and valid
if real_matches is not None and real_matches.shape[0] > 0:
    if real_matches.shape[0] > len(matches):
        matches = real_matches
        print(f"\n✓ Updated to real matches data: {matches.shape}")
    
if real_elo is not None and real_elo.shape[0] > 0:
    if real_elo.shape[0] > len(elo):
        elo = real_elo
        print(f"✓ Updated to real ELO data: {elo.shape}")

print(f"\n📊 Current data loaded:")
print(f"   Matches: {matches.shape}")
# Safely print ELO shape (guard against unbound 'elo' in static analysis / runtime)
_e = globals().get("elo", None)
if _e is None:
    print("   ELO: None")
else:
    print(f"   ELO: {_e.shape}")

Checking for real data files...

✓ Found matches file: c:\Github\fifa2026-prediction\data\raw\international_matches.csv
  ✓ Loaded with utf-8 encoding
  Shape: (47399, 9)
✓ Found ELO file: c:\Github\fifa2026-prediction\data\raw\elo_ratings.csv

📊 Current data loaded:
   Matches: (49368, 10)
   ELO: (10, 2)


## Where to Get Real FIFA Data

Here are the best sources for real international football and FIFA data:

### Option 1: Kaggle Datasets (Recommended)
- **Dataset**: "International Football Results (1872-2024)"
  - URL: https://www.kaggle.com/datasets/martj42/international-football-results-19272018
  - Download and place CSV in `data/raw/international_matches.csv`
  
- **Dataset**: "FIFA 2026 World Cup Predictions"
  - Contains player stats, team ratings, and historical data
  - URL: https://www.kaggle.com/search?q=fifa+2026

### Option 2: Football-Data.co.uk
- Historical match results: https://www.football-data.co.uk/
- Free CSV download with match results by country

### Option 3: Create Your Own CSV
Upload your CSV files with these column names:
- **Matches**: `Date`, `Home`, `Away`, `HG` (home goals), `AG` (away goals)
- **ELO**: `team`, `elo_rating`

### Option 4: API Sources
- **Football API**: https://www.football-data.org/
- **ESPN API**: For live match data
- **WhoScored**: Advanced football stats

**Next Step**: Download a dataset from Kaggle and place it in `data/raw/` folder, then run the cell below to load it.
